In [76]:
 !pip install xlrd

In [77]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher

In [78]:
import boto3
import pandas as pd
from io import StringIO, BytesIO

s3 = boto3.client("s3", region_name="us-east-1")
bucket = "food-waste-project"

def load_csv(key, **kwargs):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")), **kwargs)

def load_xls(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.ExcelFile(BytesIO(obj["Body"].read()), engine="xlrd")

# Load Instacart files
products = load_csv("raw/products.csv", usecols=["product_id", "product_name", "aisle_id", "department_id"])
aisles = load_csv("raw/aisles.csv", usecols=["aisle_id", "aisle"])
departments = load_csv("raw/departments.csv", usecols=["department_id", "department"])

# Load FoodKeeper
xls = load_xls("FoodKeeper-Data.xls")
all_sheets = {sheet: xls.parse(sheet) for sheet in xls.sheet_names}
foodkeeper = all_sheets["Product"].copy()

print("Products:", products.shape)
print("FoodKeeper:", foodkeeper.shape)

Products: (49688, 4)
FoodKeeper: (661, 38)


In [79]:
#merge instacart product metadata

instacart_df = products.merge(aisles, on="aisle_id", how="left")
instacart_df = instacart_df.merge(departments, on="department_id", how="left")

instacart_df["product_name"] = instacart_df["product_name"].fillna("").astype(str)
instacart_df["aisle"] = instacart_df["aisle"].fillna("").astype(str)
instacart_df["department"] = instacart_df["department"].fillna("").astype(str)

#keep food-only departments
food_departments = [
    "produce",
    "dairy eggs",
    "meat seafood",
    "frozen",
    "pantry",
    "bakery",
    "snacks",
    "beverages",
    "dry goods pasta",
    "canned goods",
    "breakfast",
    "deli",
    "international",
    "alcohol"
]

instacart_df = instacart_df[instacart_df["department"].isin(food_departments)].copy()

print("Instacart shape:", instacart_df.shape)

Instacart shape: (36143, 6)


In [80]:
#clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

instacart_df["product_name_clean"] = instacart_df["product_name"].apply(clean_text)
instacart_df["aisle_clean"] = instacart_df["aisle"].apply(clean_text)
instacart_df["department_clean"] = instacart_df["department"].apply(clean_text)


In [81]:
#prep foodkeper text
#since foodkeepr schemas can vary, we build a combined searchable textfiels from all object coumns
fk = foodkeeper.copy()

for col in fk.columns:
    if fk[col].dtype == "object":
        fk[col] = fk[col].fillna("").astype(str)

object_cols = fk.select_dtypes(include="object").columns.tolist()

# Combine all text columns into one searchable field
fk["combined_text"] = fk[object_cols].astype(str).agg(" | ".join, axis=1)
fk["combined_text_clean"] = fk["combined_text"].apply(clean_text)

print("FoodKeeper prepared shape:", fk.shape)


FoodKeeper prepared shape: (661, 40)


In [82]:
import pandas as pd
import numpy as np
import re

In [83]:
# Clean FoodKeeper columns
foodkeeper.columns = foodkeeper.columns.str.strip()

In [84]:
# Keep only useful columns
foodkeeper_clean = foodkeeper[[
    "Name",
    "Keywords",
    "Pantry_Max",
    "Refrigerate_Max",
    "Freeze_Max"
]].copy()


In [85]:
# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

foodkeeper_clean["Name_clean"] = foodkeeper_clean["Name"].apply(clean_text)
foodkeeper_clean["Keywords_clean"] = foodkeeper_clean["Keywords"].fillna("").apply(clean_text)


In [86]:
def get_shelf_life_days(row):
    # Prefer refrigerated shelf life
    if not pd.isna(row.get("Refrigerate_Max")):
        return float(row["Refrigerate_Max"])

    # Then pantry
    if not pd.isna(row.get("Pantry_Max")):
        return float(row["Pantry_Max"])

    # Then freeze
    if not pd.isna(row.get("Freeze_Max")):
        return float(row["Freeze_Max"])

    # Then DOP refrigerated
    if not pd.isna(row.get("DOP_Refrigerate_Max")):
        metric = str(row.get("DOP_Refrigerate_Metric", "")).lower()
        val = float(row["DOP_Refrigerate_Max"])
        if "month" in metric:
            return val * 30
        if "week" in metric:
            return val * 7
        return val

    # Then DOP pantry
    if not pd.isna(row.get("DOP_Pantry_Max")):
        metric = str(row.get("DOP_Pantry_Metric", "")).lower()
        val = float(row["DOP_Pantry_Max"])
        if "month" in metric:
            return val * 30
        if "week" in metric:
            return val * 7
        return val

    # Then DOP freeze
    if not pd.isna(row.get("DOP_Freeze_Max")):
        metric = str(row.get("DOP_Freeze_Metric", "")).lower()
        val = float(row["DOP_Freeze_Max"])
        if "month" in metric:
            return val * 30
        if "week" in metric:
            return val * 7
        return val

    # Fallback
    return 30.0

In [87]:
foodkeeper_clean["shelf_life_days"] = foodkeeper.apply(get_shelf_life_days, axis=1)
print(foodkeeper_clean.head(10))

             Name                                           Keywords  \
0          Butter                                             Butter   
1      Buttermilk                                         Buttermilk   
2          Cheese                     Cheese,cheddar, swiss,parmesan   
3          Cheese                    Cheese,parmesan,shredded,grated   
4          Cheese                 Cheese,shredded,cheddar,mozzarella   
5          Cheese               Cheese,processed slices,slices,slice   
6          Cheese                       Cheese,brie, bel paese, goat   
7  Coffee creamer  Coffee creamer,Coffee, creamer,liquid refriger...   
8  Cottage cheese                              Cottage cheese,cheese   
9    Cream cheese                                                NaN   

   Pantry_Max  Refrigerate_Max  Freeze_Max      Name_clean  \
0         NaN              NaN         NaN          butter   
1         NaN              NaN         NaN      buttermilk   
2         NaN        

instacart

In [88]:
import pandas as pd
import numpy as np
import re

In [89]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(text.split())

In [90]:
#build foodkeepr lookups
foodkeeper_clean["Name_clean"] = foodkeeper_clean["Name_clean"].fillna("").apply(clean_text)
foodkeeper_clean["Keywords_clean"] = foodkeeper_clean["Keywords_clean"].fillna("").apply(clean_text)


In [91]:
# Name lookup
name_lookup = {}
for _, row in foodkeeper_clean.iterrows():
    name = row["Name_clean"]
    days = row["shelf_life_days"]
    if name:
        if name not in name_lookup or days < name_lookup[name]:
            name_lookup[name] = days

In [92]:
# Keyword lookup
keyword_lookup = {}
for _, row in foodkeeper_clean.iterrows():
    days = row["shelf_life_days"]
    keywords = row["Keywords_clean"].split()
    for kw in keywords:
        if len(kw) > 2:
            if kw not in keyword_lookup or days < keyword_lookup[kw]:
                keyword_lookup[kw] = days

In [93]:
#token lookup
name_token_lookup = {}
for name, days in name_lookup.items():
    for token in name.split():
        if len(token) > 2:
            if token not in name_token_lookup or days < name_token_lookup[token]:
                name_token_lookup[token] = days

In [94]:
#filter to food only
products_meta = products.merge(aisles, on="aisle_id", how="left")
products_meta = products_meta.merge(departments, on="department_id", how="left")

food_departments = [
    "produce", "dairy eggs", "meat seafood", "frozen", "pantry",
    "bakery", "snacks", "beverages", "dry goods pasta",
    "canned goods", "breakfast", "deli", "international", "alcohol"
]

products_meta = products_meta[
    products_meta["department"].isin(food_departments)
].copy()

In [95]:
#matching function

def match_shelf_life_fast(product_name):
    text = str(product_name).strip().lower()
    tokens = text.split()

    # exact match
    if text in name_lookup:
        return name_lookup[text]

    # token match
    matched_days = [name_token_lookup[t] for t in tokens if t in name_token_lookup]
    if matched_days:
        return min(matched_days)

    # keyword match
    matched_days = [keyword_lookup[t] for t in tokens if t in keyword_lookup]
    if matched_days:
        return max(matched_days)

    # fallback
    return 30.0

In [96]:
#apply model
products_meta["product_name_clean"] = products_meta["product_name"].fillna("").apply(clean_text)

products_meta["shelf_life_days"] = products_meta["product_name_clean"].apply(match_shelf_life_fast)



In [97]:
def adjust_shelf_life(row):
    name = str(row["product_name"]).lower()
    dept = str(row["department"]).lower()
    aisle = str(row["aisle"]).lower()
    days = row["shelf_life_days"]

    # beverages usually shouldn't be 1-2 days unless truly fresh/refrigerated
    if dept == "beverages":
        return max(days, 14)

    # pantry / canned / dry goods should last longer
    if dept in ["pantry", "dry goods pasta", "canned goods", "breakfast"]:
        return max(days, 30)

    # frozen should definitely last longer
    if dept == "frozen":
        return max(days, 60)

    # snacks / bakery usually at least moderate shelf life
    if dept in ["snacks", "bakery"]:
        return max(days, 14)

    # common shelf-stable keywords
    if any(x in name for x in ["pasta", "rice", "quinoa", "beans", "cereal", "salt", "oil", "syrup"]):
        return max(days, 30)

    # beverages and sauces
    if any(x in name for x in ["juice", "tea", "coffee", "water", "soda", "sauce", "dressing"]):
        return max(days, 14)

    return days

products_meta["shelf_life_days"] = products_meta.apply(adjust_shelf_life, axis=1)

In [98]:
#convert to spoilage risk
def spoilage_risk_from_shelf_life(days):
    if days <= 5:
        return "high"
    elif days <= 14:
        return "medium"
    else:
        return "low"

products_meta["spoilage_risk"] = products_meta["shelf_life_days"].apply(spoilage_risk_from_shelf_life)



In [99]:
#check results
print(products_meta[["product_name", "shelf_life_days", "spoilage_risk"]].head(15))
print(products_meta["spoilage_risk"].value_counts())

print(products_meta[["product_name", "shelf_life_days", "spoilage_risk"]].sample(20, random_state=2025))

                                         product_name  shelf_life_days  \
0                          Chocolate Sandwich Cookies             14.0   
1                                    All-Seasons Salt             30.0   
2                Robust Golden Unsweetened Oolong Tea             14.0   
3   Smart Ones Classic Favorites Mini Rigatoni Wit...             60.0   
4                           Green Chile Anytime Sauce             30.0   
6                      Pure Coconut Water With Orange             14.0   
7                   Cut Russet Potatoes Steam N' Mash             60.0   
8                   Light Strawberry Blueberry Yogurt              2.0   
9      Sparkling Orange Juice & Prickly Pear Beverage             14.0   
10                                  Peach Mango Juice             14.0   
11                         Chocolate Fudge Layer Cake             60.0   
15                      Mint Chocolate Flavored Syrup             14.0   
16                                  Re

In [100]:
spoilage_df = products_meta[[
    "product_id",
    "product_name",
    "shelf_life_days",
    "spoilage_risk"
]].copy()

spoilage_df.to_csv("spoilage_predictions.csv", index=False)

print(spoilage_df.head())
print(spoilage_df.shape)

   product_id                                       product_name  \
0           1                         Chocolate Sandwich Cookies   
1           2                                   All-Seasons Salt   
2           3               Robust Golden Unsweetened Oolong Tea   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...   
4           5                          Green Chile Anytime Sauce   

   shelf_life_days spoilage_risk  
0             14.0        medium  
1             30.0           low  
2             14.0        medium  
3             60.0           low  
4             30.0           low  
(36143, 4)


In [101]:
import io

# Save to S3
csv_buffer = io.StringIO()
spoilage_df.to_csv(csv_buffer, index=False)
s3.put_object(
    Bucket=bucket,
    Key="predictions/spoilage_predictions.csv",
    Body=csv_buffer.getvalue()
)
print("Spoilage predictions saved to S3!")

Spoilage predictions saved to S3!


In [102]:
!pip install pymysql

In [105]:
import pymysql
import boto3
import pandas as pd
from io import StringIO

# Load waste predictions from S3
s3 = boto3.client("s3", region_name="us-east-1")
obj = s3.get_object(Bucket="food-waste-project", Key="predictions/waste_predictions_rds.csv")
rds_df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))
print("Loaded rows:", len(rds_df))
print(rds_df.head())

# Connect to RDS
conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin",
    password="ChurnLab2026!",
    port=3306,
    database="food_waste"
)

cursor = conn.cursor()

# Upload
for _, row in rds_df.iterrows():
    cursor.execute("""
        INSERT INTO waste_predictions 
        (product_id, product_name, predicted_waste_category, prediction_confidence, waste_risk)
        VALUES (%s, %s, %s, %s, %s)
    """, (row["product_id"], row["product_name"], row["predicted_waste_category"],
          row["prediction_confidence"], row["waste_risk"]))

conn.commit()
cursor.close()
conn.close()
print("Waste predictions uploaded!")

Loaded rows: 36143
   product_id                                       product_name  \
0           1                         Chocolate Sandwich Cookies   
1           2                                   All-Seasons Salt   
2           3               Robust Golden Unsweetened Oolong Tea   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...   
4           5                          Green Chile Anytime Sauce   

  predicted_waste_category  prediction_confidence waste_risk  
0                    snack               0.998760     medium  
1                   pantry               0.969854        low  
2                 beverage               0.999072     medium  
3                   frozen               0.999095        low  
4                   pantry               0.925072        low  
Waste predictions uploaded!
